# Installation - Python Packages

In [ ]:
!pip install pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
!pip uninstall -y dataproc-spark-connect

Found existing installation: dataproc-spark-connect 1.1.0
Uninstalling dataproc-spark-connect-1.1.0:
  Successfully uninstalled dataproc-spark-connect-1.1.0


In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/", exist_ok=True)
os.makedirs("/content/data/lab1_customers_delta", exist_ok=True)
os.makedirs("/content/data/lab1_customers_parquet", exist_ok=True)
drive.mount("/content/drive")

Mounted at /content/drive


# Creating a Spark Session for the Labs

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

def get_spark(app_name: str="DeltaLake-Labs") -> SparkSession:
    builder = (
        SparkSession.builder.appName(app_name)
        .master("local[*]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.shuffle.partitions", "8")
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()

# First Delta Table

In [ ]:
spark = get_spark("first-delta-table")
data = [(1, "Alice", "NY"), (2, "Bob", "CA"), (3, "Carol", "TX")]
df = spark.createDataFrame(data, ["id", "name", "state"])
delta_path = "/content/data/lab1_customers_delta"
parquet_path = "/content/data/lab1_customers_parquet"

# Write the data to delta
df.write.format("delta").mode("overwrite").save(delta_path)

# Write the data to parquet
df.write.format("parquet").mode("overwrite").save(parquet_path)

# Read the data from Delta Table
spark.read.format("delta").load(delta_path).show()

# Read the data from Parquet files
spark.read.format("parquet").load(parquet_path).show()

+---+-----+-----+
| id| name|state|
+---+-----+-----+
|  1|Alice|   NY|
|  2|  Bob|   CA|
|  3|Carol|   TX|
+---+-----+-----+

+---+-----+-----+
| id| name|state|
+---+-----+-----+
|  1|Alice|   NY|
|  2|  Bob|   CA|
|  3|Carol|   TX|
+---+-----+-----+



# Compare the directory of Delta & Parquet

In [ ]:
import os

print("====Delta Table Directory====")
for f in sorted(os.listdir(delta_path)):
  print(" ", f)

print("\n ====Parquet Directory====")
for f in sorted(os.listdir(parquet_path)):
  print(" ", f)

print("\n ====Delta Log Directory====")
for f in sorted(os.listdir(f"{delta_path}/_delta_log")):
  print(" ", f)

with open(f"{delta_path}/_delta_log/00000000000000000000.json") as fh:
  print(fh.read())

====Delta Table Directory====
  .part-00000-234c072e-0e42-4f71-9bb9-81871a718cac-c000.snappy.parquet.crc
  .part-00000-c54c480f-cf98-42e1-8c1e-e50a62531f27-c000.snappy.parquet.crc
  .part-00000-f6c67003-bf18-4a0d-aef8-ce59e3994873-c000.snappy.parquet.crc
  .part-00001-2aa4940c-c1c4-4d4c-86f6-d334783af383-c000.snappy.parquet.crc
  .part-00001-87d99323-ab8c-4754-bda5-db174f6cd257-c000.snappy.parquet.crc
  .part-00001-9792d5de-2bf8-4cc2-a903-8244367f95d8-c000.snappy.parquet.crc
  _delta_log
  part-00000-234c072e-0e42-4f71-9bb9-81871a718cac-c000.snappy.parquet
  part-00000-c54c480f-cf98-42e1-8c1e-e50a62531f27-c000.snappy.parquet
  part-00000-f6c67003-bf18-4a0d-aef8-ce59e3994873-c000.snappy.parquet
  part-00001-2aa4940c-c1c4-4d4c-86f6-d334783af383-c000.snappy.parquet
  part-00001-87d99323-ab8c-4754-bda5-db174f6cd257-c000.snappy.parquet
  part-00001-9792d5de-2bf8-4cc2-a903-8244367f95d8-c000.snappy.parquet

 ====Parquet Directory====
  ._SUCCESS.crc
  .part-00000-5e279f00-87ed-4c1a-ab43-fe3ce

# Transactions using DESCRIBE HISTORY

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`").select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+--------------------------------------+
|version|timestamp              |operation|operationParameters                   |
+-------+-----------------------+---------+--------------------------------------+
|2      |2026-08-02 11:23:50.093|WRITE    |{mode -> Overwrite, partitionBy -> []}|
|1      |2026-08-02 10:56:40.483|WRITE    |{mode -> Overwrite, partitionBy -> []}|
|0      |2026-08-02 10:08:23.81 |WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-----------------------+---------+--------------------------------------+



# Delta Transaction logs

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab2_orders", exist_ok=True)

In [ ]:
import json
spark = get_spark("transaction-logs")

path = "/content/data/lab2_orders"

columns = ["order_id", "amount"]
df1 = spark.createDataFrame([(1, 100.0), (2, 250.5)], columns)
df1.write.format("delta").mode("overwrite").save(path)

df2 = spark.createDataFrame([(3, 300.0), (4, 400.0)], columns)
df2.write.format("delta").mode("append").save(path)

df3 = spark.createDataFrame([(5, 299.95)], columns)
df3.write.format("delta").mode("overwrite").save(path)

def show_commit(version: int):
    fname = f"{path}/_delta_log/{version:020d}.json"
    print(f"--- Commit {version} ({fname}) ---")
    with open(fname) as fh:
        for line in fh:
            action = json.loads(line)
            action_type = list(action.keys())[0]
            details = list(action.values())[0]
            keep = {k: v for k, v in details.items()
                    if k in ("path", "schemaString", "partitionColumns", "operation")}
            print(" ", action_type, "->", keep)
for v in range(3):
    show_commit(v)

--- Commit 0 (/content/data/lab2_orders/_delta_log/00000000000000000000.json) ---
  commitInfo -> {'operation': 'WRITE'}
  metaData -> {'schemaString': '{"type":"struct","fields":[{"name":"order_id","type":"long","nullable":true,"metadata":{}},{"name":"amount","type":"double","nullable":true,"metadata":{}}]}', 'partitionColumns': []}
  protocol -> {}
  add -> {'path': 'part-00000-92184e61-fbf5-4fe3-b3de-4c15ba2b34cb-c000.snappy.parquet'}
  add -> {'path': 'part-00001-bccd1e4a-5255-499a-9a77-e419cfe46f41-c000.snappy.parquet'}
--- Commit 1 (/content/data/lab2_orders/_delta_log/00000000000000000001.json) ---
  commitInfo -> {'operation': 'WRITE'}
  add -> {'path': 'part-00000-a0cfa0bf-da02-41e2-b092-518dc1c93a22-c000.snappy.parquet'}
  add -> {'path': 'part-00001-ea4b9491-bc3d-480c-a3a0-cd79400f0a23-c000.snappy.parquet'}
--- Commit 2 (/content/data/lab2_orders/_delta_log/00000000000000000002.json) ---
  commitInfo -> {'operation': 'WRITE'}
  add -> {'path': 'part-00001-e18eaf5b-48b4-4966-

# Trace using DESCRIBE HISTORY

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+--------------------------------------+
|version|timestamp              |operation|operationParameters                   |
+-------+-----------------------+---------+--------------------------------------+
|2      |2026-08-02 11:28:24.744|WRITE    |{mode -> Overwrite, partitionBy -> []}|
|1      |2026-08-02 11:28:14.346|WRITE    |{mode -> Append, partitionBy -> []}   |
|0      |2026-08-02 11:28:13.257|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-----------------------+---------+--------------------------------------+



# ACID Transaction in Delta

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab3_accountss", exist_ok=True)

In [ ]:
spark = get_spark("acid-transactions-in-delta")
path="/content/data/lab3_accounts"

columns = ["acc_id", "balance"]
df = spark.createDataFrame([(1, 1000.00), (2, 5000.00)], columns)
df.write.format("delta").mode("overwrite").save(path)

for amt in [1100, 1200, 1300.50]:
  spark.sql(f"UPDATE delta.`{path}` SET balance = balance = {amt} WHERE acc_id = 1")

# Check the transaction logs
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
    "version", "operation", "operationParameters"
).show(truncate=False)

bad_df = spark.createDataFrame([(3, "amount-not-found")], columns)
try:
  bad_df.write.format("delta").mode("overwrite").save(path)
except Exception as e:
  print("Write rejected before commit: ", type(e).__name__)

# Verifying table consistency
spark.read.format("delta").load(path).show()
write_versions = spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select("version").show(truncate=True)


+-------+---------+--------------------------------------+
|version|operation|operationParameters                   |
+-------+---------+--------------------------------------+
|11     |UPDATE   |{predicate -> ["(acc_id#16200L = 1)"]}|
|10     |UPDATE   |{predicate -> ["(acc_id#14193L = 1)"]}|
|9      |UPDATE   |{predicate -> ["(acc_id#13073L = 1)"]}|
|8      |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|7      |UPDATE   |{predicate -> ["(acc_id#10717L = 1)"]}|
|6      |UPDATE   |{predicate -> ["(acc_id#9597L = 1)"]} |
|5      |UPDATE   |{predicate -> ["(acc_id#8477L = 1)"]} |
|4      |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|3      |UPDATE   |{predicate -> ["(acc_id#6250L = 1)"]} |
|2      |UPDATE   |{predicate -> ["(acc_id#5130L = 1)"]} |
|1      |UPDATE   |{predicate -> ["(acc_id#4009L = 1)"]} |
|0      |WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+---------+--------------------------------------+

Write rejected before commit:  AnalysisException
+-----

# Append, Overwrite and Replace

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab4_sales", exist_ok=True)

In [ ]:
from pyspark.sql import functions as F
spark = get_spark("append-overwrite-replace")
path = "/content/data/lab4_sales"

columns=["sale_date", "region", "amount"]
day1 = spark.createDataFrame(
    [("2026-01-01", "west", 100), ("2026-01-01", "east", 200)], columns
)
day1.write.format("delta").mode("overwrite").partitionBy("sale_date").save(path)

day2 = spark.createDataFrame(
    [("2026-01-02", "west", 900), ("2026-01-02", "east", 200)], columns
)
day2.write.format("delta").mode("append").save(path)

day3 = spark.createDataFrame(
    [('2026-01-03', "north", 300), ('2026-01-03', "south", 450)], columns
)
day3.write.format("delta").mode("append").save(path)

day1_updated = spark.createDataFrame(
    [("2026-01-01", "west", 120)], columns
)

# replaceWhere replaces the selected entity
day1_updated.write.format("delta") \
  .mode("overwrite") \
  .option("replaceWhere", "sale_date = '2026-01-01' AND region = 'west'") \
  .save(path)

# Idempotent Run
for _ in range(2):
  (day2.write.format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", "sale_date = '2026-01-02'")
        .save(path)
  )

  # replaceWhere
  day3_updated = spark.createDataFrame(
      [('2026-01-03', 'north', 455)], columns
  )
  day3_updated.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "sale_date = '2026-01-03' AND region = 'north'") \
    .save(path)

spark.read.format("delta").load(path).orderBy(F.col("sale_date").desc(), F.col("region").asc()).show()

+----------+------+------+
| sale_date|region|amount|
+----------+------+------+
|2026-01-03| north|   455|
|2026-01-03| south|   450|
|2026-01-02|  east|   200|
|2026-01-02|  west|   900|
|2026-01-01|  east|   200|
|2026-01-01|  west|   120|
+----------+------+------+



'\nWhy replaceWhere ?\n- overwrite is costly than replaceWhere\n- it is safer than append\n- it is idempotent in nature\n'

Why **replaceWhere**?
- overwrite is costly than replaceWhere
- it is safer than append
- it is idempotent in nature

**Append** - Adds to the end of the existing entries.
**Overwrite** - Replaces all the entries with the incoming entries.
**replaceWhere** - Replaces the selected entries on the go based on a condition.

# Schema Evolution

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab5_events", exist_ok=True)

In [ ]:
from pyspark.errors import AnalysisException
spark = get_spark("schema-evolution")
path = "/content/data/lab5_events"

columns = ["event_id", "event_type", "event_date"]
base = spark.createDataFrame(
    [(1, "click", "2026-01-01"), (2, "error", "2026-01-02")], columns
)
base.write.format("delta").mode("overwrite").save(path)

# Extra Unexpected Column
bad1 = spark.createDataFrame(
    [(2, "view", "2026-01-02", "extra_field")], ["event_id", "event_type", "event_date", "unexpected_col"]
)
try:
  bad1.write.format("delta").mode("append").save(path)
except AnalysisException as e:
  print("Rejected (extra  column):", str(e)[:200])

# Wrong type for existing column
bad2 = spark.createDataFrame(
    [(3, "view", 20260103)], columns
)
try:
  bad2.write.format("delta").mode("append").save(path)
except AnalysisException as e:
  print("Rejected (wrong type):", str(e)[:200])

fixed = bad2.withColumn("event_date", bad2.event_date.cast("string"))
fixed.write.format("delta").mode("append").save(path)

spark.read.format("delta").load(path).show()


Rejected (extra  column): [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 9b3ae39b-651c-4c67-b6de-aa22e9aa4faa).
To enable schema migration using DataFrameWriter or DataStr
Rejected (wrong type): [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'event_date' and 'event_date'
+--------+----------+----------+
|event_id|event_type|event_date|
+--------+----------+----------+
|       2|     error|2026-01-02|
|       1|     click|2026-01-01|
|       3|      view|  20260103|
+--------+----------+----------+



In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
    "version", "operation"
).show(truncate=False)

+-------+---------+
|version|operation|
+-------+---------+
|3      |WRITE    |
|2      |WRITE    |
|1      |WRITE    |
|0      |WRITE    |
+-------+---------+



# Schema Evolution

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab6_customers", exist_ok=True)

In [ ]:
spark = get_spark("schema-evolution")
path = "/content/data/lab6_customers"

schema = ["cust_id", "name"]
v1 = spark.createDataFrame([(1, "Alice"), (2, "Bob")], schema)
v1.write.format("delta").mode("overwrite").save(path)

# mergeSchema - Adding a new column to exisitng schema
new_schema = ["cust_id", "name", "user_tier"]
v2 = spark.createDataFrame([(3, "Carol", "gold")], new_schema)
v2.write.format("delta").mode("append").option("mergeSchema",  "true").save(path)

spark.read.format("delta").load(path).orderBy("user_tier").show()

+-------+-----+---------+
|cust_id| name|user_tier|
+-------+-----+---------+
|      2|  Bob|     NULL|
|      1|Alice|     NULL|
|      3|Carol|     gold|
+-------+-----+---------+



# Nested Schema Evolution

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab6_nested_customers", exist_ok=True)

In [ ]:
from pyspark.sql import Row
path="/content/data/lab6_nested_customers"
data_original = [(1, "Alice", Row(city="NYC", country="US")),
        (2, "Jhon", Row(city="Monte Carlo", country="US")),
        (3, "Ramesh", Row(city="Bangalore", country="IN")),
        (4, "Babai", Row(city="Dhaka", country="BD"))]
columns = ["id", "name", "address"]
nested_v1 = spark.createDataFrame(data_original, columns)
nested_v1.write.format("delta").mode("overwrite").save(path)

data_added = [(5, "Preeti", Row(city="Kolkata", zipcode="700002"))]
nested_v2 = spark.createDataFrame(data_added, columns)
nested_v2.write.format("delta").mode("append").option("mergeSchema", "true").save(path)

spark.read.format("delta").load(path).show()

spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
    "version", "operation", "operationParameters"
).show(truncate=False)

+---+------+--------------------+
| id|  name|             address|
+---+------+--------------------+
|  5|Preeti|{Kolkata, NULL, 7...|
|  1| Alice|     {NYC, US, NULL}|
|  2|  Jhon|{Monte Carlo, US,...|
|  3|Ramesh|{Bangalore, IN, N...|
|  4| Babai|   {Dhaka, BD, NULL}|
+---+------+--------------------+

+-------+---------+--------------------------------------+
|version|operation|operationParameters                   |
+-------+---------+--------------------------------------+
|4      |WRITE    |{mode -> Append, partitionBy -> []}   |
|3      |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|2      |WRITE    |{mode -> Append, partitionBy -> []}   |
|1      |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|0      |WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+---------+--------------------------------------+



Take Away: **Schema Enforcement** is the process in which when the entries does not match the schema the entry is rejected. **Schema Evolution** is the process in which when the entries does not match the schema, the schema get's evolved and the unmatched entries gets a **"NULL"** tag done using **mergeSchema**.

# Update and Delete Operation

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab7_customers", exist_ok=True)

In [ ]:
from delta.tables import DeltaTable
import os

spark = get_spark("update-delete")
path="/content/data/lab7_customers"

columns = ["cust_id", "name", "status"]
df = spark.createDataFrame(
    [(1, "Alice", "active"), (2, "Bob", "active"), (3, "Carol", "inactive")],
    columns)
df.write.format("delta").mode("overwrite").save(path)

dt=DeltaTable.forPath(spark, path)

# Predicate based update
dt.update(condition="cust_id=2", set={"status": "'suspended'"})

# Delete unwanted records
dt.delete("status = 'inactive'")

# Printo output
dt.toDF().show()

# Inspect the transaction log after each operation
dt.history().select("version", "operation", "operationParameters", "timestamp").show(truncate=False)

# Showing UPDATE and DELETE created new add/remove pairs, not in place edits
print(sorted(os.listdir(path)))

+-------+-----+---------+
|cust_id| name|   status|
+-------+-----+---------+
|      2|  Bob|suspended|
|      1|Alice|   active|
+-------+-----+---------+

+-------+---------+-------------------------------------------+-----------------------+
|version|operation|operationParameters                        |timestamp              |
+-------+---------+-------------------------------------------+-----------------------+
|2      |DELETE   |{predicate -> ["(status#8392 = inactive)"]}|2026-08-02 17:58:28.819|
|1      |UPDATE   |{predicate -> ["(cust_id#8390L = 2)"]}     |2026-08-02 17:58:20.439|
|0      |WRITE    |{mode -> Overwrite, partitionBy -> []}     |2026-08-02 17:58:09.072|
+-------+---------+-------------------------------------------+-----------------------+

['.part-00000-46effa02-f816-49ec-b5c9-9d9887440d3e-c000.snappy.parquet.crc', '.part-00000-487b5647-3171-406f-94a6-8f660368f821-c000.snappy.parquet.crc', '.part-00000-fe276634-4b33-4ecc-a079-c7bc4857c91a-c000.snappy.parquet.crc

# Update and Delete #Practise-2

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab7_students", exist_ok=True)

In [ ]:
import os
from delta.tables import DeltaTable
from pyspark.sql import functions as F
spark = get_spark("update-delete-practise-2")

path="/content/data/lab7_students"

data=[("1", "Alan", "CSE", 2028),
      ("2", "Amit", "DS", 2028),
      ("3", "Ronit", "IT", 2027),
      ("4", "Kriti", "CHE", 2026),
      ("5", "Danish", "EE", 2029),
      ("6", "Preeti", "ME", 2028),
      ("7", "Ashik", "EE", 2029),
      ("8", "Ayan", "ME", 2027),
      ("9", "Ashik", "AGE", 2027)]
columns = ["id", "name", "dept", "passout_yr"]
df = spark.createDataFrame(data, columns)
df.write.format("delta").mode("overwrite").save(path)

dt = DeltaTable.forPath(spark, path)

dt.update(condition=F.col("dept")=="ME", set={"passout_yr": F.lit(2028)})

dt.delete((F.col("name")=="Ashik") & (F.col("dept")=="EE") & (F.col("passout_yr")==2029))

dt.toDF().show()

dt.history().select("version", "operation", "operationParameters").show(truncate=False)

+---+------+----+----------+
| id|  name|dept|passout_yr|
+---+------+----+----------+
|  5|Danish|  EE|      2029|
|  6|Preeti|  ME|      2028|
|  8|  Ayan|  ME|      2028|
|  9| Ashik| AGE|      2027|
|  1|  Alan| CSE|      2028|
|  2|  Amit|  DS|      2028|
|  3| Ronit|  IT|      2027|
|  4| Kriti| CHE|      2026|
+---+------+----+----------+

+-------+---------+------------------------------------------------------------------------------------------------+
|version|operation|operationParameters                                                                             |
+-------+---------+------------------------------------------------------------------------------------------------+
|11     |DELETE   |{predicate -> ["(((name#21055 = Ashik) AND (dept#21056 = EE)) AND (passout_yr#21057L = 2029))"]}|
|10     |UPDATE   |{predicate -> ["(dept#21056 = ME)"]}                                                            |
|9      |WRITE    |{mode -> Overwrite, partitionBy -> []}         

# Merge (Upsert)

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab8_customers", exist_ok=True)

In [ ]:
import shutil
import os

path = "/content/data/lab8_customers"

if os.path.exists(path):
    shutil.rmtree(path)

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

path = "/content/data/lab8_customers"

schema = StructType([
    StructField("cust_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("state", StringType(), True)
])
target = spark.createDataFrame(
    [(1, "Alice", "ny"), (2, "Bob", "ca")], schema)
target.write.format("delta").mode("overwrite").save(path)

dt = DeltaTable.forPath(spark, path)

updates = spark.createDataFrame(
    [(2, "Bob", "tx"), (3, "Carol", "wa"), (3, "Carol", "wa")], schema)

# Deduplicate the source on the merge key before merging
window_specs = Window.partitionBy("cust_id").orderBy(F.lit(1))
updates_dedup = updates.withColumn("row_num", F.row_number().over(window_specs)) \
                        .filter("row_num = 1").drop("row_num")

def build_reusable_merge(delta_table, source_df, key_cols):
  key_cond = " AND ".join(f"t.{k}=s.{k}" for k in key_cols)
  delta_table.alias("t") \
    .merge(source_df.alias("s"), key_cond) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

build_reusable_merge(dt, updates_dedup, ["cust_id"])

dt.toDF().show()
dt.history().select("version", "operation", "operationParameters").show(truncate=False)

+-------+-----+-----+
|cust_id| name|state|
+-------+-----+-----+
|      1|Alice|   ny|
|      2|  Bob|   tx|
|      3|Carol|   wa|
+-------+-----+-----+

+-------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationParameters                                                                                                                                                                      |
+-------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|3      |MERGE    |{predicate -> ["(cust_id#30565 = cust_id#30571)"], matchedPredicates -> [{"actionType":"update"}], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates -> []}|
|2      |WRITE    |{m

# Merge (Upset) in Spark SQL

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab8_employees", exist_ok=True)

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

path = "/content/data/lab8_employees"

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept", StringType(), True)
])
target = spark.createDataFrame(
    [(1, "Alice", "IT"), (2, "Bob", "R&D")], schema)
target.write.format("delta").mode("overwrite").save(path)

dt = DeltaTable.forPath(spark, path)

updates = spark.createDataFrame(
    [(2, "Ruchi", "R&D"), (3, "Carol", "Management"), (3, "Carol", "Management")], schema
)

window_specs = Window.partitionBy("emp_id").orderBy(F.lit(1))
updates_dedup = updates.withColumn("rank", F.rank().over(window_specs)).filter("rank = 1").drop("rank")

# Creating temp view for SQL Query
target.createOrReplaceTempView("target_employees")
updates_dedup.createOrReplaceTempView("updates_employees")

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS employees_delta
          USING DELTA
          LOCATION '{path}'
""")

spark.sql(f"""
          MERGE INTO employees_delta AS t
          USING updates_employees AS u
          ON t.emp_id = u.emp_id
          WHEN MATCHED THEN UPDATE SET *
          WHEN NOT MATCHED THEN
            INSERT (emp_id, name, dept)
            VALUES (u.emp_id, u.name, u.dept)
""")

spark.sql("SELECT * FROM employees_delta ORDER BY emp_id").show()


+------+-----+----------+
|emp_id| name|      dept|
+------+-----+----------+
|     1|Alice|        IT|
|     2|Ruchi|       R&D|
|     3|Carol|Management|
|     3|Carol|Management|
+------+-----+----------+



# Incremental Pipeline

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab9_raw", exist_ok=True)
os.makedirs("/content/data/lab9_bronze", exist_ok=True)
os.makedirs("/content/data/lab9_silver", exist_ok=True)
os.makedirs("/content/data/lab9_gold", exist_ok=True)

In [ ]:
import shutil, os

for p in [bronze_path, silver_path, gold_path]:
    if os.path.exists(p):
        shutil.rmtree(p)

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

spark = get_spark("incremental-pipeline")
raw_path = "/content/data/lab9_raw"
bronze_path = "/content/data/lab9_bronze"
silver_path = "/content/data/lab9_silver"
gold_path = "/content/data/lab9_gold"

def write_daily_csv(day, rows):
    (spark.createDataFrame(rows, ["cust_id", "name", "state", "updated_at"])
        .write.mode("overwrite").option("header", True)
        .csv(f"{raw_path}/{day}"))
write_daily_csv("2026-01-01", [(1, "Alice", "ny", "2026-01-01T09:00:00"),
                                 (2, "Bob", "ca", "2026-01-01T09:05:00")])
write_daily_csv("2026-01-02", [(2, "Bob", "tx", "2026-01-02T10:00:00"),
                                 (3, "Carol", "wa", "2026-01-02T10:15:00")])

# Load to Bronze
def load_to_bronze(day):
  raw = (spark.read.option("header", "true").csv(f"{raw_path}/{day}").withColumn("_ingest_date", F.lit(day)))
  (raw.write.format("delta").mode("append").option("mergeSchema", "true").save(bronze_path))

# _ingest_date is the incremental marker
load_to_bronze("2026-01-01")
load_to_bronze("2026-01-02")

def bronze_to_silver(day):
  # On day one there no silver exist, so we simply create it using Day-1 data
  if not DeltaTable.isDeltaTable(spark, silver_path):
    (spark.read.format("delta").load(bronze_path).filter(F.col("_ingest_date")==day).write.format("delta").save(silver_path))
    return
  # From Day 2 we push the data in batches to the silver path
  silver = DeltaTable.forPath(spark, silver_path)
  batch = spark.read.format("delta").load(bronze_path).filter(F.col("_ingest_date")==day)
  silver.alias("s") \
    .merge(batch.alias("b"), "s.cust_id = b.cust_id") \
    .whenMatchedUpdateAll(condition="b.updated_at > s.updated_at") \
    .whenNotMatchedInsertAll() \
    .execute()
bronze_to_silver("2026-01-01")
bronze_to_silver("2026-01-02")

print("Silver table after day 1 and 2")
spark.read.format("delta").load(silver_path).orderBy("updated_at").show()

def silver_to_gold(day):
  if not DeltaTable.isDeltaTable(spark, gold_path):
    (spark.read.format("delta").load(silver_path).filter(F.col("_ingest_date")==day).write.format("delta").save(gold_path))
    return
  gold = DeltaTable.forPath(spark, gold_path)
  read_incoming_batch = spark.read.format("delta").load(silver_path).filter(F.col("_ingest_date")==day)
  gold.alias("g") \
    .merge(read_incoming_batch.alias("i"), "g.cust_id = i.cust_id") \
    .whenMatchedUpdateAll(condition="i.updated_at > g.updated_at") \
    .whenNotMatchedInsertAll() \
    .execute()
silver_to_gold("2026-01-01")
silver_to_gold("2026-01-02")

print("Gold table after day 1 and 2")
spark.read.format("delta").load(gold_path).orderBy("updated_at").show()


Silver table after day 1 and 2
+-------+-----+-----+-------------------+------------+
|cust_id| name|state|         updated_at|_ingest_date|
+-------+-----+-----+-------------------+------------+
|      1|Alice|   ny|2026-01-01T09:00:00|  2026-01-01|
|      2|  Bob|   tx|2026-01-02T10:00:00|  2026-01-02|
|      3|Carol|   wa|2026-01-02T10:15:00|  2026-01-02|
+-------+-----+-----+-------------------+------------+

Gold table after day 1 and 2
+-------+-----+-----+-------------------+------------+
|cust_id| name|state|         updated_at|_ingest_date|
+-------+-----+-----+-------------------+------------+
|      1|Alice|   ny|2026-01-01T09:00:00|  2026-01-01|
|      2|  Bob|   tx|2026-01-02T10:00:00|  2026-01-02|
|      3|Carol|   wa|2026-01-02T10:15:00|  2026-01-02|
+-------+-----+-----+-------------------+------------+



# Time Travel

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab10_timetravel", exist_ok=True)

In [ ]:
spark = get_spark("time-travel")
path = "/content/data/lab10_timetravel"

v0 = spark.createDataFrame([(1, "widget", 100)], ["sku", "name", "qty"])
v0.write.format("delta").mode("overwrite").save(path)

spark.sql(f"UPDATE delta.`{path}` SET qty = 80 WHERE sku = 1")

v2 = spark.createDataFrame([(2, "gadget", 50)], ["sku", "name", "qty"])
v2.write.format("delta").mode("append").save(path)

# Read previous versions
print("Version 0: ")
spark.read.format("delta").option("versionAsOf", 0).load(path).show()

print("Version 1: ")
spark.read.format("delta").option("versionAsOf", 1).load(path).show()

print("Current (version 2): ")
spark.read.format("delta").load(path).show()

# Compare table versions
v0_df=spark.read.format("delta").option("versionAsOf", 0).load(path)
v2_df=spark.read.format("delta").option("versionAsOf", 1).load(path)
print("Rows present now that weren't in version 0:")
v2_df.subtract(v0_df).show()

history = (spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
    "version", "operation", "operationParameters", "timestamp"
).collect())
ts_v1=str(history[-2]["timestamp"])
spark.read.format("delta").option("timestampAsOf", ts_v1).load(path).show()

Version 0: 
+---+------+---+
|sku|  name|qty|
+---+------+---+
|  1|widget|100|
+---+------+---+

Version 1: 
+---+------+---+
|sku|  name|qty|
+---+------+---+
|  1|widget| 80|
+---+------+---+

Current (version 2): 
+---+------+---+
|sku|  name|qty|
+---+------+---+
|  1|widget| 80|
|  2|gadget| 50|
+---+------+---+

Rows present now that weren't in version 0:
+---+------+---+
|sku|  name|qty|
+---+------+---+
|  1|widget| 80|
+---+------+---+

+---+------+---+
|sku|  name|qty|
+---+------+---+
|  1|widget| 80|
+---+------+---+



# Restore and Rollback

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab11_restore-rollback", exist_ok=True)

In [ ]:
spark = get_spark("restore-rollback")
path = "/content/data/lab11_restore-rollback"
good = spark.createDataFrame(
    [(1, "Widget", 9.99), (2, "Gadget", 19.99)], ["id", "item_name", "price"])
good.write.format("delta").mode("overwrite").save(path)

# Corrupt data update
spark.sql(f"UPDATE delta.`{path}` SET price=0.0 WHERE item_name='Gadget'")

print("Corrupted state: ")
spark.read.format("delta").load(path).show()

# Restore to last-known version
print("Restored State: ")
spark.sql(f"RESTORE TABLE delta.`{path}` TO VERSION AS OF 0")
spark.read.format("delta").load(path).show()

# Checking history is RESTORE operatio is logged
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

Corrupted state: 
+---+---------+-----+
| id|item_name|price|
+---+---------+-----+
|  1|   Widget| 9.99|
|  2|   Gadget|  0.0|
+---+---------+-----+

Restored State: 
+---+---------+-----+
| id|item_name|price|
+---+---------+-----+
|  2|   Gadget|19.99|
|  1|   Widget| 9.99|
+---+---------+-----+

+-------+-----------------------+---------+---------------------------------------------+
|version|timestamp              |operation|operationParameters                          |
+-------+-----------------------+---------+---------------------------------------------+
|2      |2026-08-02 22:22:22.754|RESTORE  |{version -> 0, timestamp -> NULL}            |
|1      |2026-08-02 22:22:03.262|UPDATE   |{predicate -> ["(item_name#82836 = Gadget)"]}|
|0      |2026-08-02 22:21:57.067|WRITE    |{mode -> Overwrite, partitionBy -> []}       |
+-------+-----------------------+---------+---------------------------------------------+



# Vacuum and Retention

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab12_logs", exist_ok=True)

In [ ]:
import os
spark = get_spark("vacuum-retention")
path ="/content/data/lab12_logs"

# Spark safety check - set the safety check to false (disable)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

df = spark.createDataFrame([(1, "a")], ["id", "val"])
df.write.format("delta").mode("overwrite").save(path)

for v in [2,3,4]:
  spark.createDataFrame([(v, "x")], ["id", "val"]).write.format("delta").mode("overwrite").save(path)

print("File before VACUUM:", os.listdir(path))

# VACUUM operation
spark.sql(f"VACUUM delta.`{path}` RETAIN 0 HOURS")
try:
  spark.read.format("delta").option("versionAsOf", 0).load(path)
except Exception as e:
  print("Time travel failed as: ", type(e).__name__)

# Enable the safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")


File before VACUUM: ['.part-00001-05ffd1d4-0d05-4668-870d-0189e9643e7b-c000.snappy.parquet.crc', '.part-00001-da8ab563-d045-48bb-8403-3d3ac28d2aec-c000.snappy.parquet.crc', '_delta_log', 'part-00001-da8ab563-d045-48bb-8403-3d3ac28d2aec-c000.snappy.parquet', '.part-00001-5bb3acf3-d55b-4cdc-a270-236cf1aad429-c000.snappy.parquet.crc', '.part-00000-dc47d5c7-00ea-4043-ac15-0d6336dd91b4-c000.snappy.parquet.crc', '.part-00001-d7f80b98-c3a7-4fd9-98ca-028cfa13bee7-c000.snappy.parquet.crc', 'part-00000-3fc64bcb-d5c5-4657-90c6-441e69e12366-c000.snappy.parquet', 'part-00000-0ea1f10e-ee09-4c4f-a125-8756e302bb78-c000.snappy.parquet', '.part-00000-ba396399-fc6c-4e9f-bf1d-b677153421b1-c000.snappy.parquet.crc', 'part-00001-d7f80b98-c3a7-4fd9-98ca-028cfa13bee7-c000.snappy.parquet', '.part-00000-0ea1f10e-ee09-4c4f-a125-8756e302bb78-c000.snappy.parquet.crc', '.part-00000-3fc64bcb-d5c5-4657-90c6-441e69e12366-c000.snappy.parquet.crc', 'part-00001-5bb3acf3-d55b-4cdc-a270-236cf1aad429-c000.snappy.parquet', 'p

# Parition Strategy

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab13_partition", exist_ok=True)
os.makedirs("/content/data/lab13_non-partition", exist_ok=True)

In [ ]:
from pyspark.sql import functions as F
import time, os

spark = get_spark("partition-strategy")
good_path = "/content/data/lab13_partition"
bad_path = "/content/data/lab13_non-partition"

events = (spark.range(0, 300_000)
    .withColumn("event_date",
                F.date_add(F.lit("2026-01-01"), (F.col("id") % 30).cast("int")))
    .withColumn("user_id", (F.col("id") % 1000).cast("int"))
    .withColumn("event_type", F.lit("click")))

# Good: partition by date (low cardinality, commonly filtered)
events.write.format("delta").mode("overwrite") \
    .partitionBy("event_date").save(good_path)

# Bad: partition by user_id (high cardinality -> thousands of tiny partitions)
events.write.format("delta").mode("overwrite") \
    .partitionBy("user_id").save(bad_path)

t0=time.time()
spark.read.format("delta").load(good_path).filter("event_date = '2026-01-15'").count()
print("Query on date-partition table: ", round(time.time()-t0, 2), "s")

print("Date partitions:", len(os.listdir(good_path)) - 1)
print("user_id partitions:", len(os.listdir(bad_path)) - 1)

spark.read.format("delta").load(good_path).filter("event_date = '2026-01-15'").explain(mode="formatted")


Query on date-partition table:  4.46 s
Date partitions: 30
user_id partitions: 1000
== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [id#101712L, user_id#101714, event_type#101715, event_date#101713]
Batched: true
Location: PreparedDeltaFileIndex [file:/content/data/lab13_partition]
PartitionFilters: [isnotnull(event_date#101713), (event_date#101713 = 2026-01-15)]
ReadSchema: struct<id:bigint,user_id:int,event_type:string>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [id#101712L, user_id#101714, event_type#101715, event_date#101713]

(3) Project [codegen id : 1]
Output [4]: [id#101712L, event_date#101713, user_id#101714, event_type#101715]
Input [4]: [id#101712L, user_id#101714, event_type#101715, event_date#101713]




# Small File Problem

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab14_small-file", exist_ok=True)

In [ ]:
spark = get_spark("small-file")
path = "/content/data/lab14_small-file"

# Force many tiny files by over-partitioning a small dataset before writing
spark.range(0, 200_000).repartition(400) \
    .write.format("delta").mode("overwrite").save(path)

files = [f for f in os.listdir(path) if f.endswith(".parquet")]
print("Number of data files:", len(files))
sizes_kb = [os.path.getsize(f"{path}/{f}") / 1024 for f in files]
print("Avg file size (KB):", round(sum(sizes_kb) / len(sizes_kb), 1))

# Measure query performance against the fragmented table
t0 = time.time()
spark.read.format("delta").load(path).count()
print("Count on fragmented table:", round(time.time() - t0, 3),
      "s,", len(files), "files opened")

# Analyze table metadata
spark.sql(f"DESCRIBE DETAIL delta.`{path}`") \
    .select("numFiles", "sizeInBytes").show()

Number of data files: 400
Avg file size (KB): 2.8
Count on fragmented table: 5.17 s, 400 files opened
+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|     400|    1148178|
+--------+-----------+



# Compaction (Optimization)

In [ ]:
# Previous path of Lab-14 where we had multiple small files
path = "/content/data/lab14_small-file"

In [ ]:
spark = get_spark("compaction")

# Status before Optimization
def file_stats(p):
    files = [f for f in os.listdir(p) if f.endswith(".parquet")]
    total_mb = sum(os.path.getsize(f"{p}/{f}") for f in files) / 1e6
    return len(files), total_mb
n_before, mb_before = file_stats(path)
print(f"Before: {n_before} files, {mb_before:.1f} MB total")

Before: 401 files, 2.1 MB total


In [ ]:
from delta.tables import DeltaTable

# OptionA: built-in OPTIMIZE
dt = DeltaTable.forPath(spark, path)
dt.optimize().executeCompaction()

n_after, mb_after = file_stats(path)
print(f"After OPTIMIZE: {n_after} files, {mb_after:.1f} MB total")

After OPTIMIZE: 401 files, 2.1 MB total


In [ ]:
# OptionB: Manual compaction
target_mb_per_file = 128
target_files = max(1, int(mb_after/target_mb_per_file))
spark.read.format("delta").load(path) \
  .repartition(target_files) \
  .write.format("delta") \
  .option("dataChange", "false") \
  .mode("overwrite") \
  .save(path)

n_after, mb_after = file_stats(path)
print(f"After manual compaction: {n_after} files, {mb_after:.1f} MB total")

After manual compaction: 402 files, 3.0 MB total


# Checkpoints

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab16_checkpoints", exist_ok=True)

In [ ]:
spark = get_spark("checkpoints")
path="/content/data/lab16_checkpoints"

df = spark.createDataFrame([(0, 0)], ["batch", "value"])
df.write.format("delta").mode("overwrite").save(path)

# Generating commits to exceed the checkpoint value
for i in range(1, 12):
    spark.createDataFrame([(i, i * 10)], ["batch", "value"]) \
        .write.format("delta").mode("append").save(path)

log_files = sorted(os.listdir(f"{path}/_delta_log"))
print("_delta_log contents")
for f in log_files:
  print(" ", f)

checkpoints = [f for f in log_files if "checkpoint" in f]
print("\nCheckpoint files")
for f in checkpoints:
  print(" ", f)

_delta_log contents
  .00000000000000000000.json.crc
  .00000000000000000001.json.crc
  .00000000000000000002.json.crc
  .00000000000000000003.json.crc
  .00000000000000000004.json.crc
  .00000000000000000005.json.crc
  .00000000000000000006.json.crc
  .00000000000000000007.json.crc
  .00000000000000000008.json.crc
  .00000000000000000009.json.crc
  .00000000000000000010.checkpoint.parquet.crc
  .00000000000000000010.json.crc
  .00000000000000000011.json.crc
  ._last_checkpoint.crc
  00000000000000000000.json
  00000000000000000001.json
  00000000000000000002.json
  00000000000000000003.json
  00000000000000000004.json
  00000000000000000005.json
  00000000000000000006.json
  00000000000000000007.json
  00000000000000000008.json
  00000000000000000009.json
  00000000000000000010.checkpoint.parquet
  00000000000000000010.json
  00000000000000000011.json
  _commits
  _last_checkpoint

Checkpoint files
  .00000000000000000010.checkpoint.parquet.crc
  ._last_checkpoint.crc
  00000000000000

*a table with 50,000 commits would need to parse 50,000 small JSON files just to answer “what files
make up this table right now?” Checkpoints solve this by periodically, every 10 commits by default*

# Concurrent Writes

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab17_concurrent_writes", exist_ok=True)

In [ ]:
from delta.tables import DeltaTable
import threading

spark = get_spark("concurrent-writes")
path = "/content/data/lab17_concurrent_writes"

df = spark.createDataFrame([(1, 0), (2, 0)], ["counter_id", "value"])
df.write.format("delta").mode("overwrite").save(path)

results = []
def writer(counter_id, n_increments, tag):
    local_spark = get_spark(f"Lab17-Writer-{tag}")
    dt = DeltaTable.forPath(local_spark, path)
    for _ in range(n_increments):
        try:
            dt.update(condition=f"counter_id = {counter_id}",
                      set={"value": "value + 1"})
        except Exception as e:
            results.append((tag, "conflict", str(e)[:100]))

# Two threads updating DIFFERENT rows -> no real conflict, both succeed
t1 = threading.Thread(target=writer, args=(1, 5, "row1"))
t2 = threading.Thread(target=writer, args=(2, 5, "row2"))
t1.start(); t2.start(); t1.join(); t2.join()
spark.read.format("delta").load(path).show()

# Two threads updating the SAME row concurrently -> forces real contention
t3 = threading.Thread(target=writer, args=(1, 10, "A"))
t4 = threading.Thread(target=writer, args=(1, 10, "B"))
t3.start(); t4.start(); t3.join(); t4.join()

print("Conflicts observed:", [r for r in results if r[1] == "conflict"])

spark.read.format("delta").load(path).filter("counter_id = 1").show()

spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select("version", "operation").show()


+----------+-----+
|counter_id|value|
+----------+-----+
|         2|    3|
|         1|    2|
+----------+-----+

Conflicts observed: [('row1', 'conflict', '[DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the table by a '), ('row1', 'conflict', '[DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the table by a '), ('row2', 'conflict', '[DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the table by a '), ('row1', 'conflict', '[DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the table by a '), ('row2', 'conflict', '[DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the table by a '), ('A', 'conflict', '[DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the table by a '), ('B', 'conflict', '[DELTA_CONCURRENT_APPEND] ConcurrentAppendException: Files were added to the root of the tab

# Change Data Capture

In [ ]:
from os.path import exists
from google.colab import drive
import os
os.makedirs("/content/data/lab18_change-data-capture", exist_ok=True)

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = get_spark("change-data-capture")
state_path = "/content/data/lab18_change-data-capture"

# Initial state of table
initial = spark.createDataFrame(
    [(1, "Alice", "ny"), (2, "Bob", "ca")], ["cust_id", "name", "state"])
initial.write.format("delta").mode("overwrite").save(state_path)

#  Build a CDC feed: a batch of I/U/D events, with a duplicate key
cdc_batch = spark.createDataFrame([
    (2, "Bob",   "tx", "U", "2026-01-02T10:00:00"),
    (3, "Carol", "wa", "I", "2026-01-02T10:05:00"),
    (1, "Alice", None, "D", "2026-01-02T10:10:00"),
    (2, "Bob",   "az", "U", "2026-01-02T11:00:00"),
], ["cust_id", "name", "state", "op", "event_ts"])


# Keep only the latest event per key within this batch
w = Window.partitionBy("cust_id").orderBy(F.col("event_ts").desc())
latest_per_key = (cdc_batch
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn"))

# Apply changes using MERGE, branching on the op column
dt = DeltaTable.forPath(spark, state_path)
(dt.alias("t")
    .merge(latest_per_key.alias("s"), "t.cust_id = s.cust_id")
    .whenMatchedDelete(condition="s.op = 'D'")
    .whenMatchedUpdate(condition="s.op = 'U'",
                        set={"name": "s.name", "state": "s.state"})
    .whenNotMatchedInsert(condition="s.op = 'I'",
                           values={"cust_id": "s.cust_id",
                                   "name": "s.name",
                                   "state": "s.state"})
    .execute())
print("Latest state after applying the CDC batch:")
dt.toDF().orderBy("cust_id").show()
dt.history().select("version", "operation", "operationMetrics").show(truncate=False)

Latest state after applying the CDC batch:
+-------+-----+-----+
|cust_id| name|state|
+-------+-----+-----+
|      2|  Bob|   az|
|      3|Carol|   wa|
+-------+-----+-----+

+-------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationMetrics                                                                                                                                          